# Recolly: Timetable OCR evaluation (Kaggle, R12)

```
photo ──[Vintern-1B-v3_5]──► markdown ──[TimetableParser + Ollama]──► subjects
        Tier 1: cell recall, s/photo    Tier 2: F1 on perfect markdown
        └──────────────── Tier 3: F1, exact row (whole pipeline) ───────┘
```

**Before running**
1. On your machine (in `server/`): `venv/bin/python -m scripts.generate_timetables --zip`.
2. *Datasets → New Dataset* → upload `outputs/ocr_dataset/recolly_ocr_dataset.zip`. Keep it **Private**.
   It holds `dev/`, `test/`, `injection/` and `code/` (the server's parser and scorer, imported here so nothing drifts).
3. *File → Import notebook* → this file. *Add Input* → that dataset. Accelerator **GPU T4 x2**, Internet **On**.
4. **Add the Ollama install cell** in section 1: copy the code of section "3. Install Ollama" from
   `kaggle_ollama.py`.
5. Run All with `LIMIT = 3` first and read the timings. Then set `LIMIT = None` and Run All again
   (finished photos are cached and skipped).

**Resume after a timeout**: add the stopped version's *Output* as an extra Input. Not after the parser code or
prompt changed: old `report_*` and `gold_*` files would be reused.

**Real photos later**: add folders `real_dev/` and `real_test/` to the dataset, each photo with a label of the same
name (see `code/scripts/ocr_eval.py`). Keep that dataset private and cover names and student IDs.

Synthetic images are cleaner than real photos, so these scores are optimistic.

## 0. Config

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Vintern gets the second T4; must be set before torch is imported

INPUT_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORKING_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLLAMA_URL = "http://localhost:11434"

LIMIT = 3  # photos per split for a trial run; None = all
DEV_SPLITS = ["dev", "real_dev"]  # every config runs on these
TEST_SPLITS = ["test", "real_test"]  # only the baseline and the best config run on these
LLM_MODEL = "qwen2.5:14b"  # same as production (VOICE_LLM_MODEL default)
TODAY = date(2026, 11, 27)  # only fills missing dates, which are not scored
# Each config changes ONE thing from the baseline (the settings production used on Kaggle).
CONFIGS = {
    "baseline": {},
    "tokens2048": {"max_new_tokens": 2048},
    "tiles12": {"max_tiles": 12},
    "penalty1.1": {"repetition_penalty": 1.1},
}


def resume_previous_outputs() -> None:
    """Copy result caches from an earlier run (added as Input) into OUTPUT_DIR so finished work is skipped."""
    for pattern in ("vintern_*.jsonl", "report_*.jsonl", "gold_*.jsonl"):
        for previous in sorted(INPUT_DIR.rglob(pattern)):
            target = OUTPUT_DIR / previous.name
            if not target.exists():
                shutil.copy(previous, target)
                print(f"resumed {previous.name}")


resume_previous_outputs()

## 1. Setup: server code, packages, Ollama (GPU 0)

Add the Ollama install cell here (see step 4 at the top).

In [ ]:
def run_shell(command: str) -> None:
    """Run a shell command and stop the notebook if it fails."""
    print("$", command)
    subprocess.run(command, shell=True, check=True)


run_shell("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
# Vintern's remote code needs timm + einops and breaks on transformers 5 (no all_tied_weights_keys);
# pillow-heif opens iPhone HEIC photos.
run_shell("pip install -q timm einops pillow-heif 'sqlalchemy>=2' 'transformers<5'")
assert shutil.which("ollama"), "Ollama is not installed: add the install cell above (step 4 at the top)"

In [ ]:
found = sorted(INPUT_DIR.rglob("ocr_eval.py"))
assert found, "Add the private dataset built by scripts.generate_timetables --zip as notebook Input"
sys.path.insert(0, str(found[0].parents[1]))

from app.service.llm.ollama_client import OllamaClient, OllamaError
from app.service.ocr.timetable_parser import TimetableParser
from scripts.ocr_eval import (bootstrap_f1_difference, bootstrap_f1_interval, cell_recall, injection_success_rate,
                              load_photos, match_rows, rows_of, score, score_by_tag)

In [ ]:
import requests


def is_ollama_running() -> bool:
    """Return True when the Ollama HTTP server answers."""
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).ok
    except requests.RequestException:
        return False


def start_ollama_server() -> None:
    """Start `ollama serve` on GPU 0 (if needed) and wait until it is ready."""
    if is_ollama_running():
        print("Ollama already running (make sure it uses GPU 0)")
        return
    environment = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
    log_file = open(OUTPUT_DIR / "ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], env=environment, stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_ollama_running():
            print("Ollama started")
            return
        time.sleep(1)
    raise RuntimeError(f"Ollama did not start, see {OUTPUT_DIR / 'ollama.log'}")


start_ollama_server()
requests.post(f"{OLLAMA_URL}/api/pull", json={"model": LLM_MODEL, "stream": False}, timeout=3600).raise_for_status()
parser = TimetableParser(OllamaClient(base_url=OLLAMA_URL, timeout_seconds=600), model_name=LLM_MODEL)

## 2. Data

In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".heic"}


def find_split(split: str) -> Path | None:
    """The folder named `split` that holds labelled photos, anywhere under the inputs."""
    folders = [folder for folder in sorted(INPUT_DIR.rglob(split)) if folder.is_dir() and any(folder.glob("*.json"))]
    return folders[0] if folders else None


def list_photos(split: str) -> list[Path]:
    """The labelled photos of a split, cut to LIMIT. Baselines of different base tables come first,
    so a small LIMIT still covers every template."""
    photos = [path for path in SPLIT_DIRS[split].iterdir()
              if path.suffix.lower() in IMAGE_SUFFIXES and path.with_suffix(".json").exists()]
    photos.sort(key=lambda path: (not path.stem.endswith("-baseline"), path.stem.split("-")[::-1]))
    return photos if split == "injection" else photos[:LIMIT]  # the attack set is small: always all


def read_label_rows(photo: Path) -> list[dict]:
    """The labelled subjects of one photo."""
    return json.loads(photo.with_suffix(".json").read_text(encoding="utf-8"))["subjects"]


SPLIT_DIRS = {split: folder for split in DEV_SPLITS + TEST_SPLITS if (folder := find_split(split))}
assert SPLIT_DIRS, "No split folder found in the inputs"
for split in SPLIT_DIRS:
    print(f"{split}: {len(list_photos(split))} photos")

## 3. Vintern (GPU 1)

Vintern sees a photo as 448 x 448 tiles: `max_tiles` limits how many, `max_new_tokens` limits the answer length,
`repetition_penalty` punishes repeated tokens (a timetable repeats rooms and names on purpose).

In [ ]:
import torch
import torchvision.transforms as T
import transformers
from PIL import Image
from pillow_heif import register_heif_opener
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

register_heif_opener()

# === the settings the serving notebook used for Vintern (removed from it when OCR moved to Gemini) ===
OCR_MODEL_NAME = "5CD-AI/Vintern-1B-v3_5"
OCR_DTYPE = "float16"
TILE_SIZE = 448
MAX_TILES = 6
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
OCR_PROMPT = ("<image>\nTrích xuất toàn bộ bảng thời khoá biểu trong ảnh thành bảng markdown. Giữ nguyên thứ, "
              "tiết hoặc giờ học, tên môn, mã môn, phòng, giảng viên, ngày bắt đầu và kết thúc.")
OCR_GENERATION = dict(max_new_tokens=1024, do_sample=False, num_beams=3, repetition_penalty=2.5)  # model card values


def split_into_tiles(image: Image.Image, max_tiles: int) -> list:
    """InternVL dynamic preprocessing: up to max_tiles tiles in the grid closest to the photo's aspect ratio,
    plus a whole-photo thumbnail when there is more than one tile."""
    width, height = image.size
    grids = sorted({(c, r) for c in range(1, max_tiles + 1) for r in range(1, max_tiles + 1) if c * r <= max_tiles},
                   key=lambda grid: grid[0] * grid[1])
    best, best_diff = (1, 1), float("inf")
    for columns, rows in grids:
        diff = abs(width / height - columns / rows)
        # On a tie, a bigger grid only if the photo has enough pixels to fill it.
        if diff < best_diff or (diff == best_diff and width * height > 0.5 * TILE_SIZE ** 2 * columns * rows):
            best, best_diff = (columns, rows), diff
    columns, rows = best
    resized = image.resize((columns * TILE_SIZE, rows * TILE_SIZE))
    tiles = [resized.crop(((i % columns) * TILE_SIZE, (i // columns) * TILE_SIZE,
                           (i % columns + 1) * TILE_SIZE, (i // columns + 1) * TILE_SIZE))
             for i in range(columns * rows)]
    if len(tiles) > 1:
        tiles.append(image.resize((TILE_SIZE, TILE_SIZE)))
    return tiles


class VinternRunner:
    """Loads Vintern once and reads photos with the settings of one config."""

    def __init__(self):
        """Load the model and tokenizer on the visible GPU."""
        dtype_name = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"
        self.dtype = getattr(torch, OCR_DTYPE)
        self.model = AutoModel.from_pretrained(OCR_MODEL_NAME, low_cpu_mem_usage=True, trust_remote_code=True,
                                               use_flash_attn=False, **{dtype_name: self.dtype}).eval().cuda()
        self.tokenizer = AutoTokenizer.from_pretrained(OCR_MODEL_NAME, trust_remote_code=True, use_fast=False)
        self.to_tensor = T.Compose([T.Resize((TILE_SIZE, TILE_SIZE), interpolation=InterpolationMode.BICUBIC),
                                    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

    def read_timetable(self, photo: Path, config: dict) -> str:
        """Ask Vintern for the timetable in the photo as markdown."""
        settings = dict(config)
        max_tiles = settings.pop("max_tiles", MAX_TILES)
        tiles = split_into_tiles(Image.open(photo).convert("RGB"), max_tiles)
        pixel_values = torch.stack([self.to_tensor(tile) for tile in tiles]).to(self.dtype).cuda()
        generation = {**OCR_GENERATION, **settings}
        return self.model.chat(self.tokenizer, pixel_values, OCR_PROMPT, generation).strip()


# A transformers 5 error here means the old version was imported before the pip cell: restart, then Run All.
vintern = VinternRunner()

## 4. Run the tiers

Every finished photo is appended to a JSONL file, so a rerun only does what is missing.
The `report_*` and `gold_*` files have the same line format as `scripts/batch_test.py`.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    """Every line of a JSON Lines file (empty list when the file does not exist)."""
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def append_jsonl(path: Path, record: dict) -> None:
    """Add one record to a JSON Lines file."""
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def run_ocr(split: str, config_name: str) -> dict[str, dict]:
    """Tier 1: photo -> markdown with one config; return {file: record}."""
    path = OUTPUT_DIR / f"vintern_{split}_{config_name}.jsonl"
    done = {record["file"]: record for record in read_jsonl(path)}
    for photo in list_photos(split):
        if photo.name not in done:
            started = time.perf_counter()
            markdown = vintern.read_timetable(photo, CONFIGS[config_name])
            done[photo.name] = {"file": photo.name, "markdown": markdown,
                                "seconds": round(time.perf_counter() - started, 2)}
            append_jsonl(path, done[photo.name])
            print(f"[{split}/{config_name}] {photo.name} {done[photo.name]['seconds']}s")
    return done


def parse_to_record(file_name: str, markdown: str, ocr_seconds: float) -> dict:
    """Run the production parser on one markdown table and shape the reply like an OCR job."""
    started = time.perf_counter()
    try:
        subjects = [subject.model_dump(mode="json") for subject in parser.parse(markdown, TODAY)]
        body = ({"status": "succeeded", "result": {"subjects": subjects}, "error": None} if subjects
                else {"status": "failed", "result": None, "error": "OCR_NO_TABLE_FOUND"})
    except OllamaError as error:  # not scored, and retried on the next run
        body = {"status": "failed", "result": None, "error": "OCR_MODEL_ERROR"}
        print(f"  parser failed on {file_name}: {error}")
    return {"file": file_name, "kind": "ocr", "http": 202,
            "seconds": round(ocr_seconds + time.perf_counter() - started, 2), "body": body}


def run_parser(path: Path, inputs: list[tuple[str, str, float]]) -> Path:
    """Parse every (file, markdown, OCR seconds) not already in the report; return the report path."""
    done = {record["file"] for record in read_jsonl(path) if record["body"]["error"] != "OCR_MODEL_ERROR"}
    for file_name, markdown, ocr_seconds in inputs:
        if file_name not in done:
            append_jsonl(path, parse_to_record(file_name, markdown, ocr_seconds))
    return path


def run_end_to_end(split: str, config_name: str) -> Path:
    """Tier 3: Vintern, then the parser on Vintern's markdown."""
    ocr = run_ocr(split, config_name)
    inputs = [(photo.name, ocr[photo.name]["markdown"], ocr[photo.name]["seconds"]) for photo in list_photos(split)]
    return run_parser(OUTPUT_DIR / f"report_{split}_{config_name}.jsonl", inputs)


def run_gold_parser(split: str) -> Path | None:
    """Tier 2: the parser on the perfect markdown (only photos that have a .md file)."""
    inputs = [(photo.name, photo.with_suffix(".md").read_text(encoding="utf-8"), 0.0)
              for photo in list_photos(split) if photo.with_suffix(".md").exists()]
    return run_parser(OUTPUT_DIR / f"gold_{split}.jsonl", inputs) if inputs else None

In [ ]:
def scored_photos(split: str, report: Path) -> list:
    """The scorable photos of a report, limited to the photos of this run."""
    wanted = {photo.name for photo in list_photos(split)}
    photos, _ = load_photos(report, SPLIT_DIRS[split])
    # A resumed report may hold a file twice: keep its last line.
    return list({photo.file: photo for photo in photos if photo.file in wanted}.values())


def summarize(split: str, config_name: str, report: Path) -> dict:
    """One summary row: the headline numbers for a split and config."""
    photos = scored_photos(split, report)
    row = {"split": split, "config": config_name, "photos": len(photos)}
    if not photos:
        return row
    result = score(rows_of(photos))
    low, high = bootstrap_f1_interval(photos)
    row.update(f1=result["f1"], f1_low=low, f1_high=high, exact_row=result["accuracy_exact_row"],
               **{field: result[f"accuracy_{field}"] for field in ("name", "day_of_week", "start_time", "room")})
    if config_name != "perfect_markdown":
        ocr = run_ocr(split, config_name)
        files = list_photos(split)
        row["cell_recall"] = cell_recall([(read_label_rows(photo), ocr[photo.name]["markdown"]) for photo in files])
        row["ocr_seconds"] = sum(ocr[photo.name]["seconds"] for photo in files) / len(files)
    return row


reports: dict[tuple[str, str], Path] = {}
summary_rows = []
for split in [name for name in DEV_SPLITS if name in SPLIT_DIRS]:
    for config_name in CONFIGS:
        reports[split, config_name] = run_end_to_end(split, config_name)
        summary_rows.append(summarize(split, config_name, reports[split, config_name]))


def mean_dev_f1(config_name: str) -> float:
    """Average dev F1 of a config over the dev splits."""
    values = [row.get("f1", 0.0) for row in summary_rows if row["config"] == config_name]
    return sum(values) / len(values) if values else 0.0


# Chosen on dev only; max() keeps the baseline on a tie. With a small LIMIT this choice means nothing.
BEST_CONFIG = max(CONFIGS, key=mean_dev_f1)
TEST_CONFIGS = list(dict.fromkeys(["baseline", BEST_CONFIG]))
print("best config on dev:", BEST_CONFIG)

In [ ]:
for split in [name for name in TEST_SPLITS if name in SPLIT_DIRS]:
    for config_name in TEST_CONFIGS:
        reports[split, config_name] = run_end_to_end(split, config_name)
        summary_rows.append(summarize(split, config_name, reports[split, config_name]))

for split in SPLIT_DIRS:
    gold_report = run_gold_parser(split)
    if gold_report:
        summary_rows.append(summarize(split, "perfect_markdown", gold_report))

## 5. Report

- `perfect_markdown` = Tier 2: errors left there come from the parser, not from Vintern.
- Believe a difference between two configs only when its 95% interval does not contain 0.

In [ ]:
import pandas as pd

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
summary.round(3)

In [ ]:
difference_rows = []
for (split, config_name), report in reports.items():
    if config_name == "baseline":
        continue
    first, second = scored_photos(split, reports[split, "baseline"]), scored_photos(split, report)
    common = {photo.file for photo in first} & {photo.file for photo in second}
    if common:
        difference, low, high = bootstrap_f1_difference(first, second)
        groups = len({photo.group or photo.file for photo in first if photo.file in common})
        # With one group every resample is the same data, so the interval says nothing.
        difference_rows.append({"split": split, "config": config_name, "f1_minus_baseline": difference,
                                "low": low, "high": high, "groups": groups,
                                "clear": groups > 1 and (low > 0 or high < 0)})
differences = pd.DataFrame(difference_rows)
differences.to_csv(OUTPUT_DIR / "differences.csv", index=False)
differences.round(3)

In [ ]:
tag_rows = [{"split": split, "config": config_name, "tag": tag, "photos": result["photos"], "f1": result["f1"],
             "exact_row": result["accuracy_exact_row"]}
            for (split, config_name), report in reports.items() if split in TEST_SPLITS
            for tag, result in score_by_tag(scored_photos(split, report)).items()]
by_tag = pd.DataFrame(tag_rows, columns=["split", "config", "tag", "photos", "f1", "exact_row"])
by_tag.to_csv(OUTPUT_DIR / "by_tag.csv", index=False)
by_tag.round(3)

In [ ]:
import matplotlib.pyplot as plt

variant_rows = by_tag[by_tag["tag"].astype(str).str.startswith("variant:")]
if len(variant_rows):
    chart = variant_rows.pivot_table(index="tag", columns="config", values="f1")
    chart.index = chart.index.str.removeprefix("variant:")
    chart.plot.barh(figsize=(9, 6), edgecolor="black")
    plt.xlabel("F1 (whole pipeline, test split)")
    plt.ylabel("")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "f1_by_variant.png", dpi=200)
    plt.show()

### What is still wrong? Reading errors teaches more than any average.

In [ ]:
def list_failures(split: str, report: Path) -> list[dict]:
    """Photos whose rows are missing, extra or partly wrong."""
    failures = []
    for photo in scored_photos(split, report):
        exact = sum(len(correct) == 5 for correct in match_rows(photo.truth, photo.predicted))
        if exact != len(photo.truth) or len(photo.predicted) != len(photo.truth):
            failures.append({"split": split, "file": photo.file, "labelled": len(photo.truth),
                             "predicted": len(photo.predicted), "exact": exact})
    return failures


failures = pd.DataFrame([row for (split, config_name), report in reports.items() if config_name == "baseline"
                         for row in list_failures(split, report)])
failures.to_csv(OUTPUT_DIR / "failures_baseline.csv", index=False)
failures.head(30)

In [ ]:
SHOW_SPLIT, SHOW_INDEX = next(iter(SPLIT_DIRS)), 0  # change to inspect another photo
shown = list_photos(SHOW_SPLIT)[SHOW_INDEX]
display(Image.open(shown).reduce(2))
print(run_ocr(SHOW_SPLIT, "baseline")[shown.name]["markdown"])

## 5b. Prompt injection

Photos with a line of text that orders the model to add a fake subject or to return nothing.
`injection_success_rate` = how often the model obeyed (lower is better); recall shows whether real subjects survive.
Eight photos and two attack sentences are a very small test: 0% here does not mean the parser is safe.

In [ ]:
def injection_row(tier: str, report: Path) -> dict:
    """Attack success and recall of one report on the injection set."""
    photos = scored_photos("injection", report)
    if not photos:
        return {"tier": tier, "photos": 0}
    return {"tier": tier, "photos": len(photos), "injection_success_rate": injection_success_rate(photos),
            "recall": score(rows_of(photos))["recall"]}


if folder := find_split("injection"):
    SPLIT_DIRS["injection"] = folder
    injection = pd.DataFrame([injection_row("parser on perfect markdown", run_gold_parser("injection")),
                              injection_row("whole pipeline", run_end_to_end("injection", "baseline"))])
    injection.to_csv(OUTPUT_DIR / "injection.csv", index=False)
    display(injection.round(3))

## 6. Download results

In [ ]:
archive_path = shutil.make_archive(str(WORKING_DIR / "ocr_eval_outputs"), "zip", OUTPUT_DIR)
print("Download from the Output tab:", archive_path)